In [1]:
#Programma 1 di Lavinia Rotellini, matricola 581553, a.a. 2023-2024
#Inizio con l'importare e fare il download dei moduli utili

import nltk
import re
import sklearn

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')

from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from sklearn.datasets import load_files
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text  import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

#Assegno il lemmatizzatore ad una variabile per rendere il codice più leggibile
lemmatizer = WordNetLemmatizer()

#I due corpus che utilizzerò
first_text = "OECDEdu.txt"
second_text = "mrsdalloway.txt"

#LETTURA FILE

def read_file(file_path):
  with open (file_path, "r", encoding= "utf-8") as infile:
    contents = infile.read()
  return contents 

#RACCOLTA DELLE FRASI, DEI TOKEN, POS TAGGING E LEMMATIZZAZIONE DEI TESTI

#Nella funzione sotto divido il testo in frasi e token, poi realizzo il POS tagging
def analizing(text):
  sentences = nltk.tokenize.sent_tokenize(text)
  tot_tokens = []
  tot_POS = []
  for sentence in sentences:
    tokens = nltk.tokenize.word_tokenize(sentence)
    tot_tokens.extend(tokens)
    POS = nltk.tag.pos_tag(tokens)
    tot_POS.extend(POS)
  return sentences, tot_tokens, tot_POS


#Questa seconda funzione serve a trasformare la treebank delle parts of speech in una treebank leggibile secondo WordNet
def get_wordnet_pos(treebank_tag):
  if treebank_tag.startswith('J'):
    return wordnet.ADJ

  if treebank_tag.startswith('V'):
    return wordnet.VERB

  if treebank_tag.startswith('N'):
    return wordnet.NOUN

  if treebank_tag.startswith('R'):
    return wordnet.ADV

  else:
    return wordnet.NOUN

#Con la prossima funzione eseguo la funzione di analisi e individuo i lemmi per ognuno dei due testi
def main(file_path):
  text = read_file(file_path)
  sentences, tot_tokens, tot_POS= analizing(text)
  lemmatizzazione = [lemmatizer.lemmatize(word, get_wordnet_pos(POS)) for word, POS in tot_POS]
  return sentences, tot_tokens, tot_POS, lemmatizzazione

first_sentences, first_tokens, first_POS, first_lemma = main(first_text)
second_sentences, second_tokens, second_POS, second_lemma = main(second_text)

#LEMMATIZZAZIONE, così da avere una lista di tuple in cui abbiamo il token affiancato al lemma
def lemmatizzazione(tokens, lemmi, corpus):
  lemmatizzazione = list(zip(tokens, lemmi))
  print(f"Coppie di token/lemma del {corpus} testo: {lemmatizzazione}")

lemmatizzazione(first_tokens, first_lemma, corpus = "primo")
lemmatizzazione(second_tokens, second_lemma, corpus = "secondo")


###PUNTO UNO###
#PARAGONE TRA NUMERO DI FRASI

def sentences_compared(sentences1, sentences2):
  print(f"La lunghezza del primo testo è di: {len(sentences1)} frasi.")
  print(f"La lunghezza del secondo testo è di: {len(sentences2)} frasi.")
  if len(sentences1)> len(sentences2):
    print("Il primo testo è più lungo del secondo di", len(sentences1) - len(sentences2), "frasi.")
  elif len(sentences2) > len(sentences1):
    print("Il secondo testo è più lungo del primo di", len(sentences2) - len(sentences1), "frasi.")
  else:
    print("I due file hanno lo stesso numero di frasi.")
  return 

sentences_compared(first_sentences, second_sentences)

#PARAGONE TRA NUMERO DI TOKEN
def tokens_compared(token1, token2):
  print(f"Il primo testo contiene {len(token1)} token.")
  print(f"Il secondo testo contiene {len(token2)} token.")

  if len(token1) > len(token2):
    print(f"Il primo testo è più lungo del secondo di {len(token1) - len(token2)} token.")
  elif len(token2) > len(token1):
    print(f"Il secondo testo è più lungo del primo di {len(token2) - len(token1)} token.")
  else:
    print("I due testi hanno lo stesso numero di token.")

tokens_compared(first_tokens, second_tokens)


###PUNTO DUE###
#PARAGONE TRA LA LUNGHEZZA MEDIA DELLE FRASI IN TOKEN

def ave_length_sentences(sentences1, sentences2):
  len_sentences1 = []
    
  #Tokenizzo frase per frase per contare la lunghezza in token di ogni frase ed 
  #inserirla nella variabile len_sentences1, che contiene le lunghezza di ogni frase
  for sentence in sentences1:
    tokens1 = nltk.tokenize.word_tokenize(sentence)
    len_sentences1.append(len(tokens1))        
    
  len_sentences2 = []
  for sentence in sentences2:
    tokens2 = nltk.tokenize.word_tokenize(sentence)
    len_sentences2.append(len(tokens2))
    
  #Calcolo la lunghezza media delle frasi facendo la somma delle lunghezze delle frasi diviso il numero di items nella lista len_sentences, 
  #questo numero corrisponde al numero di token nella frase
  ave_length1 = sum(len_sentences1)/len(len_sentences1)
  ave_length2 = sum(len_sentences2)/len(len_sentences2)

  #Stampo la lunghezza media delle frasi per ogni testo e il confronto

  print(f"Il primo testo ha una lunghezza media delle frasi in token di: {ave_length1:.3f}")
  print(f"Il secondo testo ha una lunghezza media delle frasi in token di: {ave_length2:.3f}")
    
  if ave_length1 > ave_length2:
    print(f"Il primo testo presenta frasi più lunghe in media di {ave_length1-ave_length2:.3f} token.")
  elif ave_length2 > ave_length1:
    print(f"Il secondo testo presenta frasi più lunghe in media di {ave_length2-ave_length1:.3f} token.")
  else:
    print("I due testi hanno la stessa lunghezza media di frasi in token")
  return

ave_length_sentences(first_sentences, second_sentences)

#PARAGONE DELLA LUNGHEZZA MEDIA DEI TOKEN
   
def ave_length_token(tokens1, tokens2):
  
  #Trasformo la lista di tokens in stringa così da poterci applicare la regex per eliminare la punteggiatura
  str1_no_punct = re.sub(r'[^\w\s]', '', str(tokens1))
  str2_no_punct = re.sub(r'[^\w\s]', '', str(tokens2))
  
  #Riprendo i token dalla stringa precedente
  tokens1_no_punct = nltk.tokenize.word_tokenize(str1_no_punct)
  tokens2_no_punct = nltk.tokenize.word_tokenize(str2_no_punct)

  #Prendo la somma dei caratteri per ogni testo
  tot_caratteri_1 = sum(len(token) for token in str1_no_punct)
  tot_caratteri_2 = sum(len(token) for token in str2_no_punct) 
  
  #Calcolo la lunghezza media in caratteri
  ave_token1 = tot_caratteri_1/len(tokens1_no_punct)
  ave_token2 = tot_caratteri_2/len(tokens2_no_punct)
  
  #Procedo al confronto
  print(f"Il primo testo ha una lunghezza media di token di {ave_token1:.3f} caratteri")
  print(f"Il secondo testo ha una lunghezza media di token di {ave_token2:.3f} caratteri")
  
  if ave_token1 > ave_token2:
    print(f"Il primo testo ha una lunghezza media di token maggiore di {(ave_token1-ave_token2):.3f} caratteri")
  elif ave_token2 > ave_token1:
    print(f"Il secondo testo ha una lunghezza media di token maggiore di {(ave_token2-ave_token1):.3f} caratteri")
  else:
    print("I due testi hanno la stessa lunghezza media dei token in caratteri")
  return

ave_length_token(first_tokens, second_tokens)


###PUNTO TRE###
#CALCOLO DEGLI HAPAX TRA I PRIMI 500, 1000, 3000 TOKENS E POI INTERO CORPUS

#Trovo gli hapax
def hapax(tokens):
  hapax_totali = 0
  for token in list(set(tokens)):
    if tokens.count(token) == 1:
      hapax_totali += 1
  return hapax_totali

#Calcolo gli hapax secondo il limite superiore specificato nell'upper_bound
def hapax_slice(tokens, corpus, upper_bound = 500):
   
  sliced_tokens = tokens[:upper_bound]
  sliced_vocab = list(set(sliced_tokens))
  
  the_Hapax = hapax(sliced_vocab)
  print(f"Il numero di hapax nei primi {upper_bound} tokens del {corpus} è {the_Hapax}")
  return the_Hapax

hapax_slice(first_tokens, corpus = "primo testo", upper_bound = 500)
hapax_slice(first_tokens, corpus = "primo testo", upper_bound = 1000)
hapax_slice(first_tokens, corpus = "primo testo", upper_bound = 3000)
hapax_slice(first_tokens, corpus = "nell'intero primo testo", upper_bound = len(first_tokens))

hapax_slice(second_tokens, corpus = "secondo testo", upper_bound = 500)
hapax_slice(second_tokens, corpus = "secondo testo", upper_bound = 1000)
hapax_slice(second_tokens, corpus = "secondo testo", upper_bound = 3000)
hapax_slice(second_tokens, corpus = "nell'intero secondo testo", upper_bound = len(second_tokens))


###PUNTO QUATTRO###
#CALCOLO TTR E VOCABOLARIO PER PORZIONI INCREMENTALI

#Uso la funzione sotto per prendere in considerazione solo i primi 200 caratteri del testo e calcolarci la TTR e il vocabolario
def TTR(tokens, upper_bound = 200):
  TTR_slice = tokens[:upper_bound]
  vocab = list(set(TTR_slice))

  TTR = len(vocab)/len(TTR_slice)
  print(f"La type/token ratio per i primi {len(TTR_slice)} è di {TTR:.3f} e il vocabolario è di: {len(vocab)}")

#Applico la funzione precedente ma con un ciclo che alzerà il numero di token presi in considerazione di volta in volta
def increm_TTR(tokens):
  step = 0
  while step < len(tokens):
    if step + 200 > len(tokens):
      step = len(tokens)
    else:
      step += 200
    TTR(tokens, upper_bound = step)

increm_TTR(first_tokens)
increm_TTR(second_tokens)


###PUNTO CINQUE###
#NUMERO DI LEMMI DISTINTI

def diff_lemma(lemmi, corpus):
  lemmi_totali = []
  for lemma in lemmi:
    if lemma in lemmi_totali:
      lemmi_totali = lemmi_totali
    else:
      lemmi_totali.append(lemma)
  print(f"Il numero di lemmi distinti per il {corpus} è di {len(lemmi_totali)}")

diff_lemma(first_lemma, corpus = "primo testo")
diff_lemma(second_lemma, corpus = "secondo testo")


###PUNTO SEI###
#DISTRIBUZIONE DI FRASI CON POLARITA' POSITIVA O NEGATIVA

#Apro e carico il corpus annotato con le classi pos e neg, mischiando le classi al suo interno
films = "movie_reviews"
films_dataset = load_files(films, shuffle=True)

#Divido il corpus in training set e test set con il metodo apposito
X_train, X_test, y_train, y_test = train_test_split(films_dataset.data, films_dataset.target, test_size = 0.20, random_state = 32)

#Associo il vettorizzatore ed il classificatore alle rispettive variabili
vectorizer = TfidfVectorizer(tokenizer=nltk.word_tokenize, max_features=3000)
classifier = MultinomialNB()

#Realizzo la pipeline con le tasks che dovrà effettuare il modello a cui sottoporremo i dati di allenamento
pipeline = Pipeline([
    ('tfidf-vectorizer', vectorizer),
    ('multinomialNB', classifier)
])

#Alleno e testo il modello
trained_model = pipeline.fit(X_train, y_train)
predicted = trained_model.predict(X_test)

#Calcolo l'accuratezza del modello con precision, recall, f1 score ecc.
classification_report(y_test, predicted, target_names = films_dataset.target_names)

#Il modello assegna la classe di appartenenza alle frasi del primo e del secondo corpus
#Trasformo le predizioni in liste così da applicarci .count alla funzione successiva

pred1 = list(trained_model.predict(first_sentences))
pred2 = list(trained_model.predict(second_sentences))

#La funzione seguente calcola la distribuzione delle frasi con polarità negativa e positiva e la porta in percentuale
def distribuzione_polarità(pred):
  pos = pred.count(0)
  neg = pred.count(1)
  
  distribuzione_pos = pos/len(pred) * 100
  distribuzione_neg = neg/len(pred) *100

  return distribuzione_pos, distribuzione_neg

#Stampiamo le distribuzioni dei due testi
distribuzione_pos1, distribuzione_neg1 = distribuzione_polarità(pred1) 
distribuzione_pos2, distribuzione_neg2 = distribuzione_polarità(pred2)

def print_distr_pol(distr, classe, corpus):
    print(f"La distribuzione delle frasi {classe} nel {corpus} testo is {distr:.2f}%.")

print_distr_pol(distribuzione_pos1, classe = "positive", corpus = "primo")
print_distr_pol(distribuzione_pos2, classe = "positive", corpus = "secondo")

print_distr_pol(distribuzione_neg1, classe = "negative", corpus = "primo")
print_distr_pol(distribuzione_neg2, classe = "negative", corpus = "secondo")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\cinna\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\cinna\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\cinna\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\cinna\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     C:\Users\cinna\AppData\Roaming\nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\cinna\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!

Coppie di token/lemma del primo testo: [('In', 'In'), ('2022', '2022'), (',', ','), ('as', 'a'), ('countries', 'country'), ('were', 'be'), ('still', 'still'), ('dealing', 'deal'), ('with', 'with'), ('the', 'the'), ('lingering', 'linger'), ('impacts', 'impact'), ('of', 'of'), ('the', 'the'), ('COVID-19', 'COVID-19'), ('pandemic', 'pandemic'), (',', ','), ('nearly', 'nearly'), ('700', '700'), ('000', '000'), ('students', 'student'), ('from', 'from'), ('81', '81'), ('OECD', 'OECD'), ('Member', 'Member'), ('and', 'and'), ('partner', 'partner'), ('economies', 'economy'), (',', ','), ('representing', 'represent'), ('29', '29'), ('million', 'million'), ('across', 'across'), ('the', 'the'), ('world', 'world'), (',', ','), ('took', 'take'), ('the', 'the'), ('Programme', 'Programme'), ('for', 'for'), ('International', 'International'), ('Student', 'Student'), ('Assessment', 'Assessment'), ('(', '('), ('PISA', 'PISA'), (')', ')'), ('test', 'test'), ('.', '.'), ('It', 'It'), ('makes', 'make'), ('2

C:\Users\cinna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


La distribuzione delle frasi positive nel primo testo is 29.59%.
La distribuzione delle frasi positive nel secondo testo is 64.81%.
La distribuzione delle frasi negative nel primo testo is 70.41%.
La distribuzione delle frasi negative nel secondo testo is 35.19%.
